# 비전 검사 파이프라인 - 교육/데모용

## 개요

로봇 비전 검사를 위한 전체 파이프라인을 단계별로 실행합니다.

### 파이프라인

```
Mesh → Viewpoints → IK Solutions → Trajectory → Collision Check
 .obj     .h5          .h5          .csv          .csv
```

### 학습 목표

1. FOV 기반 뷰포인트 샘플링
2. 역기구학(IK) 및 충돌 검사  
3. TSP + 동적 프로그래밍
4. 적응형 보간 및 충돌 체크

## Section 0: 환경 설정

In [1]:
import sys
print(sys.executable)

import torch
print(torch.__version__)

/isaac-sim/kit/python/bin/python3
2.7.0+cu128


In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Isaac Sim 환경 설정
os.environ['LD_LIBRARY_PATH'] = '/isaac-sim/kit/python/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
sys.path.insert(0, '/isaac-sim/kit/python/lib/python3.11/site-packages')

import numpy as np
import h5py
from tqdm.notebook import tqdm
import open3d as o3d
import torch

# 프로젝트 모듈
PROJECT_ROOT = Path('/isaac-sim/curobo/vision_inspection')
sys.path.insert(0, str(PROJECT_ROOT))

from common import config as cfg
from common.cli_utils import print_section_header, print_key_value, print_success
from common.data_io import save_viewpoints_hdf5, load_viewpoints_hdf5

print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Open3D: {o3d.__version__}")
print("\n✅ 환경 설정 완료")

In [3]:
# 파이프라인 설정
OBJECT_ID = 500
MESH_FILE = cfg.PROJECT_ROOT / 'data' / 'object' / 'glass.obj'

VIEWPOINT_DIR = cfg.DATA_ROOT / 'viewpoint' / str(OBJECT_ID)
IK_DIR = cfg.DATA_ROOT / 'ik' / str(OBJECT_ID)
TRAJECTORY_DIR = cfg.DATA_ROOT / 'trajectory' / str(OBJECT_ID)

for d in [VIEWPOINT_DIR, IK_DIR, TRAJECTORY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 알고리즘 파라미터
CURVATURE_WEIGHT = 1.5
MIN_TILT_ANGLE_DEG = 30.0
KNN_NEIGHBORS = 10

print_section_header("설정 완료")
print(f"객체 ID: {OBJECT_ID}")
print(f"메쉬: {MESH_FILE}")


설정 완료
객체 ID: 500
메쉬: /isaac-sim/curobo/vision_inspection/data/object/glass.obj


---
## Section 1: Mesh → Viewpoints

FOV 기반 적응형 샘플링으로 뷰포인트 생성

In [4]:
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from mesh_to_viewpoints import (
    load_mesh_file, compute_surface_curvature,
    estimate_required_viewpoints, sample_points_adaptive_poisson,
    filter_downward_facing_viewpoints, apply_minimum_tilt_angle,
    CameraSpec, Viewpoint  # CameraSpec, Viewpoint 클래스 import 추가
)

print_section_header("1. 메쉬 로딩")
mesh, surface_area = load_mesh_file(str(MESH_FILE))
print(f"Vertices: {len(mesh.vertices):,}")
print(f"Triangles: {len(mesh.triangles):,}")
print(f"Area: {surface_area * 1e6:.2f} mm²")

# CameraSpec 객체 생성
camera_spec = CameraSpec(
    fov_width_mm=cfg.CAMERA_FOV_WIDTH_MM,
    fov_height_mm=cfg.CAMERA_FOV_HEIGHT_MM,
    working_distance_mm=cfg.CAMERA_WORKING_DISTANCE_MM,
    overlap_ratio=cfg.CAMERA_OVERLAP_RATIO
)

print_section_header("2. 뷰포인트 수 추정")
# 올바른 파라미터로 함수 호출
num_vp = estimate_required_viewpoints(
    mesh, camera_spec, target_coverage=1.0, curvature_weight=CURVATURE_WEIGHT
)
print(f"추정 뷰포인트: {num_vp}")

print_section_header("3. 샘플링 및 필터링")
# sample_points_adaptive_poisson은 내부에서 curvature를 계산함
points, normals = sample_points_adaptive_poisson(
    mesh, 
    num_vp, 
    curvature_weight=CURVATURE_WEIGHT,
    base_overlap_ratio=cfg.CAMERA_OVERLAP_RATIO
)

# numpy array를 Viewpoint 객체 리스트로 변환
viewpoints = [Viewpoint(position=p, normal=n) for p, n in zip(points, normals)]
print(f"초기 뷰포인트: {len(viewpoints)}")

# 필터링 (Viewpoint 리스트를 받음)
# viewpoints, num_removed = filter_downward_facing_viewpoints(viewpoints)
# print(f"하향 뷰포인트 제거: {num_removed}개 제거, {len(viewpoints)}개 남음")

viewpoints, num_adjusted = apply_minimum_tilt_angle(viewpoints, camera_spec, MIN_TILT_ANGLE_DEG)
print(f"최소 틸트 각도 적용: {num_adjusted}개 조정")
print(f"최종 뷰포인트: {len(viewpoints)}")

# Viewpoint 리스트를 numpy array로 다시 변환 (저장용)
points = np.array([vp.position for vp in viewpoints])
normals = np.array([vp.normal for vp in viewpoints])

# 저장 (파라미터 순서 수정)
vp_file = VIEWPOINT_DIR / 'viewpoints.h5'
save_viewpoints_hdf5(
    points,           # positions가 첫 번째
    normals,          # normals가 두 번째
    str(vp_file),     # output_path가 세 번째
    camera_spec={
        'fov_width_mm': cfg.CAMERA_FOV_WIDTH_MM,
        'fov_height_mm': cfg.CAMERA_FOV_HEIGHT_MM,
        'working_distance_mm': cfg.CAMERA_WORKING_DISTANCE_MM,
        'overlap_ratio': cfg.CAMERA_OVERLAP_RATIO
    }
)
print_success(f"✅ Section 1 완료: {len(points)} viewpoints → {vp_file}")


1. 메쉬 로딩
Loading mesh from: /isaac-sim/curobo/vision_inspection/data/object/glass.obj
Loaded mesh: 461 vertices, 876 triangles
Surface area: 151379.20 mm² (assuming mesh is in meters)

Mesh coordinate range (Z-up coordinate system):
  X: [-0.180307, 0.180339] (range: 0.360646)
  Y: [-0.183993, 0.186064] (range: 0.370057)
  Z: [-0.002256, 0.096989] (range: 0.099245) ← up direction

✓ Mesh coordinates appear to be in METERS (max range: 0.3701m)
✓ Using Z-up coordinate system (compatible with Isaac Sim / URDF / Pinocchio)
Vertices: 461
Triangles: 876
Area: 151379.20 mm²

2. 뷰포인트 수 추정

Computing surface curvature for adaptive estimation...

Automatic viewpoint estimation (adaptive overlap):
  Surface area: 151379.20 mm²
  FOV: 41.0 × 30.0 mm
  Curvature weight: 1.50
  Adaptive overlap range: 50% (flat) → 95% (curved)
  Average overlap: 68.2%
  Average curvature: 0.1425
  Max curvature: 0.3519
  Basic estimate (uniform): 493 viewpoints
  Adaptive estimate: 1972 viewpoints
  Increase: 300.0

In [5]:
# 3D 시각화 생략 (Open3D 시각화는 Isaac Sim 환경에서 제한됨)
print(f"\n뷰포인트 생성 완료: {len(points)}개")


뷰포인트 생성 완료: 1972개


---
## Section 2: Viewpoints → IK Solutions

EAIK로 역기구학 계산 및 충돌 검사

In [6]:
# Section 2 imports
from scripts.compute_ik_solutions import (
    ComputeConfig, ViewpointManager, 
    load_viewpoints_file, create_viewpoints_from_file,
    setup_collision_world_for_ik, setup_ik_solver,
    save_ik_solutions_hdf5
)

print_section_header("1. 뷰포인트 로딩")
surface_positions, surface_normals, metadata = load_viewpoints_hdf5(str(vp_file))
print(f"뷰포인트 수: {len(surface_positions)}")

# ComputeConfig 생성
ik_cfg = ComputeConfig(
    viewpoints_path=str(vp_file),
    output_path=None,
    robot_config_file=cfg.DEFAULT_ROBOT_CONFIG
)

print_section_header("2. CuRobo 환경 설정")
world_cfg = setup_collision_world_for_ik(ik_cfg)

print_section_header("3. IK Solver 초기화")
ik_solver = setup_ik_solver(ik_cfg, world_cfg)


1. 뷰포인트 로딩
Loaded 1972 viewpoints from /isaac-sim/curobo/vision_inspection/data/viewpoint/500/viewpoints.h5
뷰포인트 수: 1972

2. CuRobo 환경 설정

SETTING UP COLLISION WORLD

Setting up collision world...
  Table: pos=[ 1.     0.    -0.425], dims=[0.6 1.  0.5]
  Wall: pos=[-1.1  0.   0.5], dims=[0.1 2.2 1. ]
  Workbench: pos=[ 0.35 -1.1   0.5 ], dims=[3.  0.1 1. ]
  Robot mount: pos=[ 0.    0.   -0.25], dims=[0.3 0.3 0.5]
  Mesh 0: /isaac-sim/curobo/vision_inspection/data/object/glass.obj at pos=[ 1.     0.    -0.172]
  Total obstacles: 4 cuboids + 1 meshes

3. IK Solver 초기화

INITIALIZING IK SOLVER
  Robot config                       : ur20_safe.yml

IK solver configuration:
  Rotation threshold                 : 0.05 rad
  Position threshold                 : 0.005 m
  Number of seeds                    : 20
  Self collision check               : True
  Collision checker                  : MESH
  Using CUDA graph                   : True
✓ IK solver initialized successfully



In [7]:
from common.ik_utils import compute_ik_eaik, assign_ik_solutions_to_viewpoints, check_ik_solutions_collision
from time import perf_counter

print_section_header("4. IK 계산 (EAIK)")

# ViewpointManager 생성
viewpoint_mgr = create_viewpoints_from_file(surface_positions, surface_normals, metadata, ik_cfg)

# Glass 위치 변환 행렬 (identity rotation)
glass_world_pose = np.eye(4, dtype=np.float64)
glass_world_pose[:3, 3] = ik_cfg.glass_position

# World poses 업데이트
viewpoint_mgr.update_world_poses(glass_world_pose)

# World matrices 수집
world_mats, used_indices = viewpoint_mgr.collect_world_matrices()
print(f"\n유효한 world poses: {len(world_mats)}")

# IK 계산
print("\nComputing IK solutions with EAIK...")
ik_start = perf_counter()
ik_results = compute_ik_eaik(world_mats)
assign_ik_solutions_to_viewpoints(viewpoint_mgr.viewpoints, ik_results, used_indices)
ik_elapsed = perf_counter() - ik_start

# 충돌 검사
print("Checking collision constraints...")
collision_start = perf_counter()
check_ik_solutions_collision(viewpoint_mgr.viewpoints, ik_solver)
collision_elapsed = perf_counter() - collision_start

# 통계
total = len(viewpoint_mgr.viewpoints)
with_all = viewpoint_mgr.count_with_all_ik()
with_safe = viewpoint_mgr.count_with_safe_ik()

print(f"\n{'='*60}")
print("IK 계산 완료")
print(f"{'='*60}")
print(f"IK 계산 시간: {ik_elapsed*1000:.2f} ms")
print(f"충돌 검사 시간: {collision_elapsed*1000:.2f} ms")
print(f"\n전체 뷰포인트: {total}")
print(f"IK 솔루션 있음: {with_all}/{total} ({with_all/total*100:.1f}%)")
print(f"충돌 없는 솔루션: {with_safe}/{total} ({with_safe/total*100:.1f}%)")

# HDF5 저장
ik_output = IK_DIR / 'ik_solutions.h5'
save_ik_solutions_hdf5(viewpoint_mgr.viewpoints, str(ik_output), str(vp_file))
print_success(f"✅ Section 2 완료: IK solutions → {ik_output}")


4. IK 계산 (EAIK)

VIEWPOINTS DATA LOADING
  Loaded viewpoints                  : 1972
  Coordinate system                  : Z-up

Coordinate ranges:
  X range                            : [-0.1793, 0.1790]
  Y range                            : [-0.1827, 0.1852]
  Z range                            : [-0.0023, 0.1882]

Using working distance from HDF5: 110.0 mm

Generating viewpoint poses...
  Offsetting by 110.0 mm along surface normals
Generated 1972 viewpoints

=== Coordinate Transform Debug ===
Object world pose:
[[ 1.     0.     0.     1.   ]
 [ 0.     1.     0.     0.   ]
 [ 0.     0.     1.    -0.172]
 [ 0.     0.     0.     1.   ]]
Local pose (Z-up):
[[ 0.1854605   0.97057315  0.15359739 -0.11207346]
 [ 0.98265172 -0.18318086 -0.02898916  0.01916463]
 [-0.          0.15630908 -0.98770819  0.20518894]
 [ 0.          0.          0.          1.        ]]
World pose result:
[[ 0.1854605   0.97057315  0.15359739  0.88792654]
 [ 0.98265172 -0.18318086 -0.02898916  0.01916463]
 [ 0. 

### IK 솔루션 통계

# IK 솔루션 수 분포
ik_counts = [len(vp.all_ik_solutions) for vp in viewpoint_mgr.viewpoints]
safe_counts = [len(vp.safe_ik_solutions) for vp in viewpoint_mgr.viewpoints]

print(f"\nIK 솔루션 통계:")
print(f"  평균 IK 솔루션 수: {np.mean(ik_counts):.2f}")
print(f"  평균 안전한 IK 솔루션 수: {np.mean(safe_counts):.2f}")
print(f"  최대 IK 솔루션 수: {np.max(ik_counts)}")
print(f"  최소 IK 솔루션 수: {np.min(ik_counts)}")

---
## Section 3: IK → Trajectory (GTSP + DP)

GTSP(Generalized TSP)로 방문 순서를 결정하고, 동적 프로그래밍으로 각 뷰포인트의 최적 IK 솔루션을 선택합니다.

In [8]:
# Section 3: Import trajectory generation functions
from scripts.fk_gtsp_gpu_claude2 import (
    build_clusters_from_h5,
    build_neighbors_knn,
    build_visit_order_robot_cost,
    choose_ik_given_order,
    export_to_csv
)

print_section_header("Section 3: IK → Trajectory (GTSP + DP)")
print("함수 임포트 완료")

[INFO] GPU acceleration enabled (CuPy detected)

Section 3: IK → Trajectory (GTSP + DP)
함수 임포트 완료


In [9]:
print_section_header("1. IK 솔루션 로딩 및 클러스터 구성")

# HDF5에서 IK 솔루션 로딩 및 FK 계산
clusters, target_coords, nonempty_map = build_clusters_from_h5(
    str(ik_output),
    use_safe_only=True,  # 충돌 없는 IK 솔루션만 사용
    tool_z=0.0
)

print(f"전체 뷰포인트: {len(viewpoint_mgr.viewpoints)}")
print(f"유효한 IK 솔루션이 있는 뷰포인트: {len(clusters)}")
print(f"제거된 뷰포인트 (IK 솔루션 없음): {len(viewpoint_mgr.viewpoints) - len(clusters)}")

# 클러스터당 IK 솔루션 수 통계
cluster_sizes = [len(c['q']) for c in clusters]
print(f"\n클러스터 통계:")
print(f"  평균 IK 솔루션 수: {np.mean(cluster_sizes):.2f}")
print(f"  최대 IK 솔루션 수: {np.max(cluster_sizes)}")
print(f"  최소 IK 솔루션 수: {np.min(cluster_sizes)}")


1. IK 솔루션 로딩 및 클러스터 구성
전체 뷰포인트: 1972
유효한 IK 솔루션이 있는 뷰포인트: 1038
제거된 뷰포인트 (IK 솔루션 없음): 934

클러스터 통계:
  평균 IK 솔루션 수: 3.98
  최대 IK 솔루션 수: 4
  최소 IK 솔루션 수: 2


In [10]:
print_section_header("2. k-NN 이웃 그래프 구성")

# k-NN 그래프 생성
nbrs = build_neighbors_knn(target_coords, k=KNN_NEIGHBORS)

# 그래프 통계
neighbor_counts = [len(nbr_list) for nbr_list in nbrs]
print(f"k-NN 그래프 통계 (k={KNN_NEIGHBORS}):")
print(f"  평균 이웃 수: {np.mean(neighbor_counts):.2f}")
print(f"  최대 이웃 수: {np.max(neighbor_counts)}")
print(f"  최소 이웃 수: {np.min(neighbor_counts)}")


2. k-NN 이웃 그래프 구성
k-NN 그래프 통계 (k=10):
  평균 이웃 수: 10.00
  최대 이웃 수: 10
  최소 이웃 수: 10


In [11]:
print_section_header("3. GTSP 방문 순서 결정")

# Greedy TSP로 방문 순서 계산 (로봇 운동학 고려)
from time import perf_counter
tsp_start = perf_counter()
order = build_visit_order_robot_cost(
    clusters,
    nbrs,
    lam_rot=0.1,  # 회전 비용 가중치
    tool_z=0.0
)
tsp_elapsed = perf_counter() - tsp_start

print(f"TSP 계산 시간: {tsp_elapsed*1000:.2f} ms")
print(f"방문 순서 길이: {len(order)} 뷰포인트")
print(f"시작 뷰포인트 인덱스: {order[0]}")
print(f"종료 뷰포인트 인덱스: {order[-1]}")


3. GTSP 방문 순서 결정
TSP 계산 시간: 177.67 ms
방문 순서 길이: 1038 뷰포인트
시작 뷰포인트 인덱스: 102
종료 뷰포인트 인덱스: 26


In [12]:
print_section_header("4. 동적 프로그래밍으로 최적 IK 선택")

# DP로 각 뷰포인트의 최적 IK 솔루션 선택
dp_start = perf_counter()
picked, total_cost = choose_ik_given_order(
    clusters,
    order,
    lam_rot=0.1,  # 회전 비용 가중치
    tool_z=0.0
)
dp_elapsed = perf_counter() - dp_start

print(f"DP 계산 시간: {dp_elapsed*1000:.2f} ms")
print(f"총 궤적 비용 (관절 공간): {total_cost:.3f}")
print(f"선택된 IK 솔루션 수: {len(picked)}")

# 각 뷰포인트마다 하나의 IK 솔루션 선택됨
assert len(picked) == len(order), "모든 뷰포인트에 대해 IK 솔루션이 선택되어야 함"


4. 동적 프로그래밍으로 최적 IK 선택
DP 계산 시간: 36.86 ms
총 궤적 비용 (관절 공간): 21.930
선택된 IK 솔루션 수: 1038


In [13]:
print_section_header("5. 궤적 CSV 저장")

# CSV 파일로 저장
gtsp_csv = TRAJECTORY_DIR / 'gtsp.csv'
export_to_csv(str(gtsp_csv), order, picked, clusters)

print_success(f"✅ Section 3 완료: GTSP trajectory → {gtsp_csv}")
print(f"\n총 Section 3 시간: {(tsp_elapsed + dp_elapsed)*1000:.2f} ms")


5. 궤적 CSV 저장
✓ ✅ Section 3 완료: GTSP trajectory → /isaac-sim/curobo/vision_inspection/data/trajectory/500/gtsp.csv

총 Section 3 시간: 214.54 ms


---
## Section 4: Trajectory → Collision Check

적응형 보간과 CuRobo를 사용하여 궤적의 충돌을 검사하고 수정합니다.

In [ ]:
# Section 4: Import collision checking modules
from scripts.curobo_check import CuRoboCollisionChecker
from common.data_io import load_trajectory_csv, save_trajectory_csv

print_section_header("Section 4: Trajectory → Collision Check")
print("함수 임포트 완료")

In [15]:
print_section_header("1. CuRobo 충돌 검사기 초기화")

# 충돌 검사기 생성 (기존 world_cfg와 ik_solver 재사용)
try:
    # CuRobo collision checker 생성
    collision_checker = CuRoboCollisionChecker(
        robot_config_path=cfg.DEFAULT_ROBOT_CONFIG,
        obstacle_mesh_paths=[str(MESH_FILE)],
        glass_position=cfg.GLASS_POSITION,
        glass_rotation=cfg.GLASS_ROTATION,
        table_position=cfg.TABLE_POSITION,
        table_dimensions=cfg.TABLE_DIMENSIONS,
        wall_position=cfg.WALL_POSITION,
        wall_dimensions=cfg.WALL_DIMENSIONS,
        workbench_position=cfg.WORKBENCH_POSITION,
        workbench_dimensions=cfg.WORKBENCH_DIMENSIONS,
        robot_mount_position=cfg.ROBOT_MOUNT_POSITION,
        robot_mount_dimensions=cfg.ROBOT_MOUNT_DIMENSIONS,
        collision_margin=cfg.COLLISION_MARGIN,
        world_cfg=world_cfg,  # 기존에 생성된 world_cfg 재사용
    )
    print("✅ 충돌 검사기 초기화 완료")
except Exception as e:
    print(f"⚠️  충돌 검사기 초기화 실패: {e}")
    print("기본 설정으로 재시도...")
    collision_checker = CuRoboCollisionChecker(
        robot_config_path=cfg.DEFAULT_ROBOT_CONFIG,
        obstacle_mesh_paths=[str(MESH_FILE)],
        glass_position=cfg.GLASS_POSITION,
        collision_margin=cfg.COLLISION_MARGIN
    )

Creating new Mesh cache: 1



1. CuRobo 충돌 검사기 초기화

INITIALIZING CUROBO COLLISION CHECKER
  Robot config                       : ur20_safe.yml
  Glass position                     : [ 1.     0.    -0.172]
  Glass rotation (quat)              : [1. 0. 0. 0.]
  Collision margin                   : 0.0 m
  Device                             : cuda:0
Using provided world configuration

Initializing MotionGen for trajectory replanning...
  MotionGen configuration:
    Interpolation dt: 0.02
    Trajectory optimization timesteps: 32
    Timeout: 8.0s
    Max attempts: 3
  ✓ MotionGen initialized
✓ CuRobo collision checker initialized

✅ 충돌 검사기 초기화 완료


In [16]:
print_section_header("2. 궤적 로딩 및 충돌 검사")

# CSV에서 궤적 로딩
trajectory, joint_names = load_trajectory_csv(str(gtsp_csv))
print(f"궤적 웨이포인트 수: {len(trajectory)}")
print(f"관절 수: {len(joint_names)}")

# 충돌 검사 수행 (적응형 보간 + 배치 충돌 검사 + 재계획)
print("\n충돌 검사 실행 중...")
collision_start = perf_counter()
results = collision_checker.check_trajectory(
    trajectory,
    adaptive_max_joint_step_deg=cfg.COLLISION_ADAPTIVE_MAX_JOINT_STEP_DEG,  # 5.0도
    exclude_last_joint=cfg.COLLISION_INTERP_EXCLUDE_LAST_JOINT,  # True
    attempt_replan=cfg.REPLAN_ENABLED,  # attempt_replan으로 수정
    max_replan_iterations=3,
    verbose=True
)
collision_time = perf_counter() - collision_start

print(f"\n충돌 검사 완료: {collision_time:.2f}초")


2. 궤적 로딩 및 충돌 검사
Loaded trajectory: 1038 waypoints, 6 joints
Joint names: ['ur20-shoulder_pan_joint', 'ur20-shoulder_lift_joint', 'ur20-elbow_joint', 'ur20-wrist_1_joint', 'ur20-wrist_2_joint', 'ur20-wrist_3_joint']
궤적 웨이포인트 수: 1038
관절 수: 6

충돌 검사 실행 중...

TRAJECTORY CHECKING - CUROBO
Input: 1038 waypoints
Replanning enabled: True
Interpolation: enabled (adaptive)
Device: cuda:0

[STEP 1] Interpolating trajectory (CPU linear)...
  Trajectory: 1038 waypoints
  Interpolation: 26772 configs
  Adaptive mode: max joint step 0.20 deg (last joint excluded)
    Steps → min 2, max 1343, avg 25.82
  Interpolation completed in 0.045s

[STEP 2] Checking collisions on interpolated trajectory...
  Total configurations to check: 27,810
  Using batched CuRobo collision checking (GPU-accelerated)
  Collision check completed in 0.053s
  Found 2283 collision(s) (0 waypoint, 2283 segment)

[STEP 3] Motion Replanning (Original Trajectory)...
  Replanning enabled: True
  Max iterations: 3

  Total segments

In [17]:
print_section_header("3. 충돌 통계 분석")

# 충돌 결과 출력
print(f"\n{'='*60}")
print("충돌 검사 결과")
print(f"{'='*60}")
print(f"웨이포인트 충돌 수: {results.get('num_collisions', 0)}")
print(f"세그먼트 충돌 수: {results.get('num_segment_collisions', 0)}")
print(f"총 충돌 수: {results.get('total_collisions', 0)}")
print(f"검사한 설정 수: {results.get('configs_checked', 0)}")

# 충돌률 계산
if results.get('configs_checked', 0) > 0:
    collision_rate = (results.get('total_collisions', 0) / results.get('configs_checked', 1)) * 100
    print(f"충돌률: {collision_rate:.2f}%")

# 재계획 통계 (있는 경우)
if 'replan_success_count' in results:
    print(f"\n재계획 통계:")
    print(f"  성공: {results.get('replan_success_count', 0)}")
    print(f"  실패: {results.get('replan_fail_count', 0)}")
    print(f"  총 시도: {results.get('replan_iterations_performed', 0)}회")
    
# 최종 궤적 정보
final_traj = results.get('final_trajectory', trajectory)
print(f"\n최종 궤적 웨이포인트 수: {len(final_traj)}")


3. 충돌 통계 분석

충돌 검사 결과
웨이포인트 충돌 수: 0
세그먼트 충돌 수: 2283
총 충돌 수: 2283
검사한 설정 수: 0

재계획 통계:
  성공: 4
  실패: 0
  총 시도: 0회

최종 궤적 웨이포인트 수: 1158


In [18]:
print_section_header("4. 충돌 없는 궤적 저장")

# 최종 충돌 없는 궤적 CSV로 저장
collision_free_csv = TRAJECTORY_DIR / 'gtsp_collision_free.csv'
save_trajectory_csv(final_traj, str(collision_free_csv), joint_names=joint_names)

# 충돌 보고서 저장
try:
    report_path = collision_checker.save_collision_report(
        str(gtsp_csv),
        results,
        timing_info={'collision_check_time': collision_time}
    )
    print(f"충돌 보고서 저장: {report_path}")
except Exception as e:
    print(f"충돌 보고서 저장 실패: {e}")

print_success(f"✅ Section 4 완료: Collision-free trajectory → {collision_free_csv}")
print(f"\n총 Section 4 시간: {collision_time:.2f}초")


4. 충돌 없는 궤적 저장
Saved trajectory: /isaac-sim/curobo/vision_inspection/data/trajectory/500/gtsp_collision_free.csv
  Waypoints: 1158, Joints: 6
충돌 보고서 저장 실패: 'CuRoboCollisionChecker' object has no attribute 'save_collision_report'
✓ ✅ Section 4 완료: Collision-free trajectory → /isaac-sim/curobo/vision_inspection/data/trajectory/500/gtsp_collision_free.csv

총 Section 4 시간: 1.83초


In [19]:
print_section_header("🎉 파이프라인 완료")

print("\n" + "="*80)
print("비전 검사 파이프라인 - 전체 요약")
print("="*80)

print("\n✅ Section 1: Mesh → Viewpoints")
print(f"   출력: {vp_file}")
print(f"   뷰포인트 수: {len(points)}")

print("\n✅ Section 2: Viewpoints → IK Solutions")
print(f"   출력: {ik_output}")
print(f"   전체 뷰포인트: {total}")
print(f"   IK 솔루션 있음: {with_all}/{total} ({with_all/total*100:.1f}%)")
print(f"   충돌 없는 솔루션: {with_safe}/{total} ({with_safe/total*100:.1f}%)")

print("\n✅ Section 3: IK → Trajectory (GTSP + DP)")
print(f"   출력: {gtsp_csv}")
print(f"   방문 순서 길이: {len(order)} 뷰포인트")
print(f"   총 궤적 비용: {total_cost:.3f}")

print("\n✅ Section 4: Trajectory → Collision Check")
print(f"   출력: {collision_free_csv}")
print(f"   초기 웨이포인트: {len(trajectory)}")
print(f"   최종 웨이포인트: {len(final_traj)}")
print(f"   충돌률: {collision_rate:.2f}%" if results.get('configs_checked', 0) > 0 else "   충돌률: N/A")

print("\n" + "="*80)
print("생성된 파일:")
print("="*80)
print(f"1. 뷰포인트:         {vp_file}")
print(f"2. IK 솔루션:        {ik_output}")
print(f"3. 초기 궤적:        {gtsp_csv}")
print(f"4. 충돌 없는 궤적:   {collision_free_csv}")

print("\n✨ 모든 단계가 성공적으로 완료되었습니다! ✨")


🎉 파이프라인 완료

비전 검사 파이프라인 - 전체 요약

✅ Section 1: Mesh → Viewpoints
   출력: /isaac-sim/curobo/vision_inspection/data/viewpoint/500/viewpoints.h5
   뷰포인트 수: 1972

✅ Section 2: Viewpoints → IK Solutions
   출력: /isaac-sim/curobo/vision_inspection/data/ik/500/ik_solutions.h5
   전체 뷰포인트: 1972
   IK 솔루션 있음: 1972/1972 (100.0%)
   충돌 없는 솔루션: 1038/1972 (52.6%)

✅ Section 3: IK → Trajectory (GTSP + DP)
   출력: /isaac-sim/curobo/vision_inspection/data/trajectory/500/gtsp.csv
   방문 순서 길이: 1038 뷰포인트
   총 궤적 비용: 21.930

✅ Section 4: Trajectory → Collision Check
   출력: /isaac-sim/curobo/vision_inspection/data/trajectory/500/gtsp_collision_free.csv
   초기 웨이포인트: 1038
   최종 웨이포인트: 1158
   충돌률: N/A

생성된 파일:
1. 뷰포인트:         /isaac-sim/curobo/vision_inspection/data/viewpoint/500/viewpoints.h5
2. IK 솔루션:        /isaac-sim/curobo/vision_inspection/data/ik/500/ik_solutions.h5
3. 초기 궤적:        /isaac-sim/curobo/vision_inspection/data/trajectory/500/gtsp.csv
4. 충돌 없는 궤적:   /isaac-sim/curobo/vision_inspection/data/t